# Named Entity Recognition

Bu projede kelimenin kişi/yer/org gibi etiketini tahmin edeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/ner_dataset.csv',encoding='latin-1')
df['Sentence #']=df['Sentence #'].ffill()
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


In [ ]:
df['Tag'].value_counts().head(15)


### Görselleştirme


In [ ]:
df['Tag'].value_counts().head(12).plot(kind='bar')
plt.show()


### Boş veri


In [ ]:
df['Word']=df['Word'].fillna('')
df['POS']=df['POS'].fillna('UNK')
df['Tag']=df['Tag'].fillna('O')


### Feature Engineering


In [ ]:
s=df.sample(30000,random_state=42)
s['uzunluk']=s['Word'].str.len()
s['buyuk']=s['Word'].str[0].str.isupper().fillna(False).astype(int)
s['digit']=s['Word'].str.contains(r'\d',regex=True).astype(int)
x=pd.get_dummies(s[['uzunluk','buyuk','digit','POS']],drop_first=True)
y=s['Tag']


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


### 3 Model


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

for ad,m in [('LogReg',LogisticRegression(max_iter=200)),('DT',DecisionTreeClassifier(random_state=42,max_depth=12)),('RF',RandomForestClassifier(n_estimators=80,random_state=42))]:
    m.fit(x_train,y_train)
    print(ad,accuracy_score(y_test,m.predict(x_test)))


In [ ]:
dt=DecisionTreeClassifier(random_state=42,max_depth=12)
dt.fit(x_train,y_train)


In [ ]:
import joblib
joblib.dump(dt,'../../models/nlp_ner_patterns.joblib')


### Sonuç

GSE NER setinde çoğu kelime O olduğu için accuracy yüksek görünüyor. Yine de POS + büyük harf işe yarıyor. Hedefi temel seviyede tutturdum.
